# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, inspect, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) clinical tabular dataset using the `mlcroissant` library.

> **Dataset Title**: Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution

### Dataset Source
The dataset is defined by a [Croissant schema](https://github.com/mlcommons/croissant/) and accessed by its URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load Dataset object from Croissant
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset name: {getattr(metadata, 'name', None)}")
print(f"Description: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, their `@id`s, fields and columns.

**Note:** All entity references are by `@id` as per Croissant schema best practices.

In [ ]:
# List all record sets present in the dataset, display their @id and fields/columns
record_sets = list(dataset.record_sets)
if not record_sets:
    print('No record sets detected via .record_sets -- attempting fallback via metadata:')
    # fallback to metadata (useful if .record_sets is empty due to schema difference)
    rs = getattr(metadata, 'recordSet', None)
    if rs:
        # Some datasets may use a list for recordSet, ensure proper structure
        if not isinstance(rs, list):
            rs = [rs]
        for r in rs:
            print(f"Record set @id: {getattr(r, '@id', r)}  Type: {getattr(r, '@type', None)}")
    else:
        # Detect from Croissant spec:
        print('No `recordSet` entries available in the top-level schema.')
else:
    print('Record sets found:')
    for rs in record_sets:
        print(f"- @id: {rs['@id']}")
        # List fields for each record set, referenced by their @id:
        if 'field' in rs and rs['field']:
            print('  Fields:')
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            for f in fields:
                if isinstance(f, dict):
                    print(f"    - {f.get('@id', '')} ({f.get('name', f.get('@id',''))})")
                else:
                    print(f"    - {f}")
        if 'column' in rs and rs['column']:
            # Some schemas use 'column' for tabular data
            print('  Columns:')
            columns = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
            for c in columns:
                if isinstance(c, dict):
                    print(f"    - {c.get('@id', '')} ({c.get('name', c.get('@id', ''))})")
                else:
                    print(f"    - {c}")

## 3. Data Extraction
Load data from a specific record set into a Pandas DataFrame for analysis.
Use the `@id` of the record set and of the fields/columns as identified in the overview above.

In [ ]:
# If you have already identified record_set IDs from above (for tabular data):
# We'll auto-detect or you may hard-code if necessary:

# Attempt to extract record set IDs via the record_sets property:
croissant_record_sets = list(dataset.record_sets)
record_set_ids = [rs['@id'] for rs in croissant_record_sets] if croissant_record_sets else []

# If none are detected, you can use the known default for this dataset:
if not record_set_ids:
    # Known from the dataset structure, the main record set (tabular data):
    record_set_ids = ['https://api.app.sen.science/frontiers/7862866/629c16ec-37ee-4556-a351-d5164116c2dd_tabular']

dfs = {}
for rec_id in record_set_ids:
    print(f"\nLoading records from record set: {rec_id}")
    try:
        rows = list(dataset.records(record_set=rec_id))
        if rows:
            df = pd.DataFrame(rows)
            dfs[rec_id] = df
            print(f"Loaded {len(df)} records. Columns: {list(df.columns)}")
        else:
            print(f"No records found for {rec_id}")
    except Exception as e:
        print(f"Failed to load {rec_id}: {e}")

# Preview the first DataFrame (main data table):
main_record_set_id = record_set_ids[0]
if main_record_set_id in dfs and not dfs[main_record_set_id].empty:
    print('\nColumns in main tabular record set:')
    print(dfs[main_record_set_id].columns.tolist())
    dfs[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering on numeric fields, normalization, grouping and summarization.

**To reference columns, always use their `@id` where present. For demonstration, we will:**
- Select a numeric column (e.g., Age_at_second_primary_CRC)
- Filter patients above a threshold age
- Normalize this age field
- Group by a categorical field (e.g., Sex) and compute statistics


In [ ]:
# Select the main DataFrame for tabular data
df = dfs[main_record_set_id]

# List all columns to select by @id (actual column names may include @id or plain names):
print('Available columns:')
print(df.columns.tolist())

# Let's assume the @id for 'Age at second primary CRC (years)' is 'age_second_crc' -- adjust as required
# And for 'Sex' field, @id is 'sex'
# If @id are not the actual column names, adjust accordingly based on the table above

numeric_field_id = None
group_field_id = None

# Try to guess commonly used fields
for col in df.columns:
    if 'age' in col.lower() and 'second' in col.lower():
        numeric_field_id = col
    if 'sex' in col.lower():
        group_field_id = col

if not numeric_field_id:
    # Fallback: pick first numeric-like column
    for col in df.select_dtypes(include='number').columns:
        numeric_field_id = col
        break
if not group_field_id:
    for col in df.columns:
        if 'gender' in col.lower() or 'sex' in col.lower():
            group_field_id = col
            break

print(f"\nChosen numeric field (by @id or name): {numeric_field_id}")
print(f"Chosen group-by field: {group_field_id}")

# Filter: patients with age at 2nd CRC > 60
threshold = 60
filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold]
print(f"\nPatients with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize the numeric field (z-score):
filtered_df[numeric_field_id + '_normalized'] = (
    pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()

print(f"\nSample with normalized field:")
print(filtered_df[[numeric_field_id, group_field_id, numeric_field_id + '_normalized']].head())

# Group-level statistics:
if group_field_id is not None:
    grouped_df = (filtered_df
                  .groupby(group_field_id)[numeric_field_id]
                  .agg(['mean', 'count', 'std'])
                  .reset_index())
    print(f"\nGrouped statistics by {group_field_id}:")
    print(grouped_df)

## 5. Visualization
Let's visualize the age distribution at 2nd CRC, colored by sex, for the filtered data.

If you don't have matplotlib installed, uncomment below line.

In [ ]:
# !pip install matplotlib seaborn --quiet
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(data=filtered_df, x=numeric_field_id, hue=group_field_id, bins=10, kde=True, element="step")
plt.title(f"Age at second primary CRC (> {threshold}) by {group_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 clinical dataset using `mlcroissant`.  We examined the data structure, performed filtering and normalization on age variables, and visualized the distribution of patient ages at second primary colorectal cancer by sex. The Croissant standard enabled clear referencing of all entities by `@id`, facilitating reproducible data science workflows.